In [ ]:
from typing import Any
from typing import Dict

import gymnasium as gym
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from collections import deque
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.env_util import make_vec_env
import torch
import numpy as np

ENV_ID = "SabreSwapEnv"

gym.register(
    id=ENV_ID, entry_point="utils.sabre_env_wog_wc:SabreSwapEnv"
)

N_TRIALS = 2000
N_STARTUP_TRIALS = 10
N_EVALUATIONS = 5
N_TIMESTEPS = int(1e6)
EVAL_FREQ = int(N_TIMESTEPS / N_EVALUATIONS)
N_EVAL_EPISODES = 5

DEFAULT_HYPERPARAMS = {
    "policy": "MlpPolicy",
    "env": make_vec_env(ENV_ID, n_envs=8, env_kwargs={"qubit_range": (10, 20), "start_level": 1}),
}


d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\envs\registration.py:736: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes ([]).
  logger.warn(


In [2]:
class CurriculumCallback(BaseCallback):
    """Optimized callback for curriculum learning."""

    def __init__(self, max_level: int = 1000, success_threshold: float = 0.8, verbose: int = 0, max_window: int = 100):
        super().__init__(verbose)
        self.max_level = max_level
        self.max_window = max_window
        self.success_threshold = success_threshold
        
        self.level = 1
        self.episode_results = deque(maxlen=max_window)  # 자동 크기 제한
        self.success_count = 0
        
        # 로깅 주기 설정 (매 스텝이 아닌 주기적으로)
        self.log_freq = 100
        self.step_count = 0

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", [])
        
        # 에피소드 완료된 환경들만 처리
        if any(dones):
            successed = self.training_env.env_method("is_success")
            
            for i, done in enumerate(dones):
                if done:
                    if len(self.episode_results) == self.max_window:
                        # 가장 오래된 결과 제거 시 success_count 업데이트
                        if self.episode_results[0]:
                            self.success_count -= 1
                    
                    # 새 결과 추가
                    success = successed[i]
                    self.episode_results.append(success)
                    if success:
                        self.success_count += 1
            
            # 레벨 업 체크
            if len(self.episode_results) >= self.max_window:
                success_rate = self.success_count / len(self.episode_results)
                if success_rate >= self.success_threshold:
                    self.level = min(self.level + 1, self.max_level)
                    self.training_env.env_method("set_level", level=self.level)
                    
                    if self.verbose > 0:
                        print(f"Level increased to {self.level} (success rate: {success_rate:.3f})")
        
        # 주기적으로만 상세 로깅
        self.step_count += 1
        if self.step_count % self.log_freq == 0:
            episode_count = len(self.episode_results)
            success_rate = self.success_count / max(1, episode_count)
            
            self.logger.record("success_rate", success_rate)
            self.logger.record("level", self.level)
            
            # 환경 상태는 덜 자주 로깅
            if self.step_count % (self.log_freq * 5) == 0:
                front_layer_len = self.training_env.env_method("front_layer_size")
                swap_candidate_len = self.training_env.env_method("swap_candidate_size")
                reset_failed = self.training_env.env_method("get_reset_failed")
                
                self.logger.record("front_layer_size/mean", np.mean(front_layer_len))
                self.logger.record("swap_candidate_size/mean", np.mean(swap_candidate_len))
                self.logger.record("reset_failed/mean", np.mean(reset_failed))
        
        return True


In [3]:

def sample_ppo_params(trial: optuna.Trial) -> Dict[str, Any]:
    """Sampler for PPO hyperparameters."""
    gamma: float = 1.0 - trial.suggest_float("gamma", 0.0001, 0.1, log=True)
    max_grad_norm: float = trial.suggest_float("max_grad_norm", 0.3, 5.0, log=True)
    gae_lambda: float = 1.0 - trial.suggest_float("gae_lambda", 0.001, 0.1, log=True)
    n_steps: int = 2 ** trial.suggest_int("exponent_n_steps", 3, 11)
    learning_rate: float = trial.suggest_float("lr", 1e-5, 1, log=True)
    ent_coef: float = trial.suggest_float("ent_coef", 0.0000001, 0.1, log=True)
    vf_coef: float = trial.suggest_float("vf_coef", 0.000001, 1.0, log=True)

    # Display true values.
    trial.set_user_attr("gamma_", gamma)
    trial.set_user_attr("gae_lambda_", gae_lambda)
    trial.set_user_attr("n_steps", n_steps)


    return {
        "n_steps": n_steps,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "learning_rate": learning_rate,
        "ent_coef": ent_coef,
        "max_grad_norm": max_grad_norm,
        "vf_coef": vf_coef,
    }


class TrialEvalCallback(EvalCallback):
    """Callback used for evaluating and reporting a trial."""

    def __init__(
        self,
        eval_env: gym.Env,
        trial: optuna.Trial,
        n_eval_episodes: int = 5,
        eval_freq: int = 10000,
        deterministic: bool = True,
        verbose: int = 0,
    ):
        super().__init__(
            eval_env=eval_env,
            n_eval_episodes=n_eval_episodes,
            eval_freq=eval_freq,
            deterministic=deterministic,
            verbose=verbose,
        )
        self.trial = trial
        self.eval_idx = 0
        self.is_pruned = False

    def _on_step(self) -> bool:
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            super()._on_step()
            self.eval_idx += 1
            self.trial.report(self.last_mean_reward, self.eval_idx)
            # Prune trial if need.
            if self.trial.should_prune():
                self.is_pruned = True
                return False
        return True


def objective(trial: optuna.Trial) -> float:
    kwargs = DEFAULT_HYPERPARAMS.copy()
    # Sample hyperparameters.
    kwargs.update(sample_ppo_params(trial))
    # Create the RL model.
    model = PPO(**kwargs, device="cuda", tensorboard_log="./optuna_sb3_wog/",verbose=1)
    # Create env used for evaluation.
    eval_env = Monitor(gym.make(ENV_ID, qubit_range=(10, 20)))
    # Create the callback that will periodically evaluate and report the performance.
    eval_callback = TrialEvalCallback(
        eval_env, trial, n_eval_episodes=N_EVAL_EPISODES, eval_freq=EVAL_FREQ, deterministic=True
    )

    nan_encountered = False
    try:
        model.learn(N_TIMESTEPS, callback=[eval_callback, CurriculumCallback(verbose=1)])
    except AssertionError as e:
        # Sometimes, random hyperparams can generate NaN.
        print(e)
        nan_encountered = True
    finally:
        # Free memory.
        model.env.close()
        eval_env.close()

    # Tell the optimizer that the trial failed.
    if nan_encountered:
        return float("nan")

    if eval_callback.is_pruned:
        raise optuna.exceptions.TrialPruned()

    return eval_callback.last_mean_reward



# Set pytorch num threads to 1 for faster training.
torch.set_num_threads(1)

sampler = TPESampler(n_startup_trials=N_STARTUP_TRIALS)
# Do not prune before 1/3 of the max budget is used.
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=N_EVALUATIONS // 3)

study = optuna.create_study(sampler=sampler, pruner=pruner, direction="maximize")
try:
    study.optimize(objective, n_trials=N_TRIALS, timeout= 60 * 60 * 4)  # 4 hours
except KeyboardInterrupt:
    pass

print("Number of finished trials: ", len(study.trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

print("  User attrs:")
for key, value in trial.user_attrs.items():
    print("    {}: {}".format(key, value))

[I 2025-06-28 01:44:35,602] A new study created in memory with name: no-name-75c516ce-56eb-42f2-a386-403796cf0506


Using cuda device
Logging to ./optuna_sb3_wog/PPO_1


d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be int32, actual type: float32
  logger.warn(
d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logge

-----------------------------------
| front_layer_size/    |          |
|    mean              | 4.62     |
| level                | 1        |
| reset_failed/        |          |
|    mean              | 1.12     |
| rollout/             |          |
|    ep_len_mean       | 51.5     |
|    ep_rew_mean       | -36.8    |
| success_rate         | 0        |
| swap_candidate_size/ |          |
|    mean              | 19.4     |
| time/                |          |
|    fps               | 599      |
|    iterations        | 1        |
|    time_elapsed      | 6        |
|    total_timesteps   | 4096     |
-----------------------------------
-----------------------------------------
| front_layer_size/       |             |
|    mean                 | 4.62        |
| level                   | 1           |
| reset_failed/           |             |
|    mean                 | 1.12        |
| rollout/                |             |
|    ep_len_mean          | 71.6        |
|    ep_rew_mean

[I 2025-06-28 02:32:27,987] Trial 0 finished with value: -inf and parameters: {'gamma': 0.004631996491801579, 'max_grad_norm': 0.45657028032110814, 'gae_lambda': 0.012989000469292447, 'exponent_n_steps': 9, 'lr': 0.004428810463879856, 'ent_coef': 0.031926572803657745, 'vf_coef': 0.21125136500738548}. Best is trial 0 with value: -inf.


Using cuda device
Logging to ./optuna_sb3_wog/PPO_2
---------------------------------
| level              | 1        |
| rollout/           |          |
|    ep_len_mean     | 17.1     |
|    ep_rew_mean     | -14.9    |
| success_rate       | 0        |
| time/              |          |
|    fps             | 476      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 1024     |
---------------------------------
------------------------------------------
| level                   | 1            |
| rollout/                |              |
|    ep_len_mean          | 40.5         |
|    ep_rew_mean          | -24.2        |
| success_rate            | 0            |
| time/                   |              |
|    fps                  | 403          |
|    iterations           | 2            |
|    time_elapsed         | 5            |
|    total_timesteps      | 2048         |
| train/                  |              |
|    approx_kl          

[W 2025-06-28 02:35:18,555] Trial 1 failed with parameters: {'gamma': 0.012812891156842228, 'max_grad_norm': 0.6272140321880103, 'gae_lambda': 0.015251685906232028, 'exponent_n_steps': 7, 'lr': 7.407260023007398e-05, 'ent_coef': 0.013063256627722203, 'vf_coef': 0.711282711823532} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "d:\lab\circuit_route\.venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\psw04\AppData\Local\Temp\ipykernel_11940\540031546.py", line 78, in objective
    model.learn(N_TIMESTEPS, callback=[eval_callback, CurriculumCallback(verbose=1)])
  File "d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\ppo\ppo.py", line 311, in learn
    return super().learn(
           ^^^^^^^^^^^^^^
  File "d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py", line 337, in learn


Number of finished trials:  2
Best trial:
  Value:  -inf
  Params: 
    gamma: 0.004631996491801579
    max_grad_norm: 0.45657028032110814
    gae_lambda: 0.012989000469292447
    exponent_n_steps: 9
    lr: 0.004428810463879856
    ent_coef: 0.031926572803657745
    vf_coef: 0.21125136500738548
  User attrs:
    gamma_: 0.9953680035081984
    gae_lambda_: 0.9870109995307076
    n_steps: 512
